In [1]:
# Cellule 1: Upload du fichier CSV
from google.colab import files
print("📁 Sélectionnez le fichier 'sales_clean_with_features.csv'")
uploaded = files.upload()

📁 Sélectionnez le fichier 'sales_clean_with_features.csv'


Saving sales_clean_with_features.csv to sales_clean_with_features.csv


In [2]:
# ============================================
# 📊 ANALYSE EXPLORATOIRE — Marché Immobilier Français
# ============================================
# Données: DVF (Demandes de Valeurs Foncières)
# Auteur: Walid Bourhaleb
# ============================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Charger les données
df = pd.read_csv('sales_clean_with_features.csv', low_memory=False)

# Convertir les dates
df['date_mutation'] = pd.to_datetime(df['date_mutation'], errors='coerce')

print(f"📊 Dataset chargé: {len(df):,} ventes, {len(df.columns)} colonnes")
print(f"📅 Période: {df['date_mutation'].min()} → {df['date_mutation'].max()}")
print(f"\n🏠 Types de biens:")
print(df['type_local'].value_counts().to_string())
print(f"\n📍 Départements:")
if 'departement' in df.columns:
    print(df['departement'].value_counts().to_string())

📊 Dataset chargé: 16,184 ventes, 49 colonnes
📅 Période: 2023-01-02 00:00:00 → 2023-12-06 00:00:00

🏠 Types de biens:
type_local
Appartement    10822
Maison          5362

📍 Départements:
departement
59.0    3275
33.0    3049
69.0    2248
13.0    1833
75.0      71


In [3]:
# ============================================
# 📋 STATISTIQUES DESCRIPTIVES
# ============================================

cols_stats = ['valeur_fonciere', 'prix_m2', 'surface_reelle_bati', 'nb_pieces']
stats = df[cols_stats].describe().round(0)
print("📊 Statistiques principales:\n")
print(stats.to_string())

print(f"\n💶 Prix médian: {df['valeur_fonciere'].median():,.0f} €")
print(f"📐 Prix/m² médian: {df['prix_m2'].median():,.0f} €/m²")
print(f"📏 Surface médiane: {df['surface_reelle_bati'].median():,.0f} m²")

📊 Statistiques principales:

       valeur_fonciere  prix_m2  surface_reelle_bati  nb_pieces
count          16184.0  15640.0              16183.0    16183.0
mean          390230.0   5318.0                 70.0        3.0
std           583659.0   3627.0                 41.0        2.0
min            10000.0    500.0                  9.0        0.0
25%           165000.0   2799.0                 42.0        2.0
50%           260000.0   4132.0                 65.0        3.0
75%           418512.0   6860.0                 89.0        4.0
max          9800000.0  20000.0                930.0       54.0

💶 Prix médian: 260,000 €
📐 Prix/m² médian: 4,132 €/m²
📏 Surface médiane: 65 m²


In [4]:
# ============================================
# 💰 DISTRIBUTION DES PRIX
# ============================================

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Distribution des prix de vente", "Distribution prix/m²"))

# Prix de vente
fig.add_trace(
    go.Histogram(x=df['valeur_fonciere'].dropna(), nbinsx=50,
                 marker_color='#3b82f6', name='Prix'),
    row=1, col=1
)

# Prix au m²
fig.add_trace(
    go.Histogram(x=df['prix_m2'].dropna(), nbinsx=50,
                 marker_color='#22c55e', name='Prix/m²'),
    row=1, col=2
)

fig.update_layout(
    title="💰 Distribution des prix immobiliers",
    height=400, showlegend=False,
    template='plotly_white'
)
fig.update_xaxes(title_text="Prix (€)", row=1, col=1)
fig.update_xaxes(title_text="Prix/m² (€)", row=1, col=2)
fig.show()

In [5]:
# ============================================
# 🏠 PRIX PAR TYPE DE BIEN
# ============================================

df_types = df[df['type_local'].isin(['Appartement', 'Maison'])].copy()

fig = px.box(df_types, x='type_local', y='prix_m2',
             color='type_local',
             color_discrete_map={'Appartement': '#3b82f6', 'Maison': '#22c55e'},
             title='🏠 Prix/m² par type de bien')

fig.update_layout(height=500, template='plotly_white', showlegend=False)
fig.update_yaxes(title='Prix/m² (€)', range=[0, 15000])
fig.show()

# Stats par type
print("\n📊 Statistiques par type:")
stats_type = df_types.groupby('type_local').agg({
    'valeur_fonciere': ['count', 'mean', 'median'],
    'prix_m2': ['mean', 'median'],
    'surface_reelle_bati': ['mean', 'median']
}).round(0)
print(stats_type.to_string())


📊 Statistiques par type:
            valeur_fonciere                     prix_m2         surface_reelle_bati       
                      count      mean    median    mean  median                mean median
type_local                                                                                
Appartement           10822  390941.0  240000.0  6091.0  4674.0                56.0   52.0
Maison                 5362  388794.0  300000.0  3794.0  3326.0                99.0   91.0


In [6]:
# ============================================
# 📍 PRIX PAR DÉPARTEMENT
# ============================================

if 'departement' in df.columns:
    dept_stats = df.groupby('departement').agg({
        'prix_m2': 'median',
        'valeur_fonciere': 'median',
        'id': 'count'
    }).rename(columns={'id': 'nb_ventes'}).sort_values('prix_m2', ascending=True)

    fig = px.bar(dept_stats.reset_index(),
                 x='prix_m2', y='departement',
                 orientation='h',
                 color='prix_m2',
                 color_continuous_scale='RdYlGn_r',
                 title='📍 Prix médian au m² par département',
                 labels={'prix_m2': 'Prix/m² médian (€)', 'departement': 'Département'})

    fig.update_layout(height=400, template='plotly_white')
    fig.show()

    print("\n📊 Détail par département:")
    print(dept_stats.to_string())


📊 Détail par département:
              prix_m2  valeur_fonciere  nb_ventes
departement                                      
59.0          2706.60         195000.0       3275
69.0          3663.37         251833.0       2248
33.0          3961.54         261750.0       3049
13.0          4083.33         266670.0       1833
75.0         12175.00         509575.0         71


In [7]:
# ============================================
# 📈 ÉVOLUTION TEMPORELLE DES PRIX
# ============================================

df['mois_str'] = df['date_mutation'].dt.to_period('M').astype(str)

monthly = df.groupby('mois_str').agg({
    'prix_m2': 'median',
    'valeur_fonciere': 'median',
    'id': 'count'
}).rename(columns={'id': 'nb_ventes'}).reset_index()

fig = make_subplots(rows=2, cols=1,
    subplot_titles=("Prix/m² médian mensuel", "Nombre de ventes mensuel"),
    shared_xaxes=True)

fig.add_trace(
    go.Scatter(x=monthly['mois_str'], y=monthly['prix_m2'],
               mode='lines+markers', name='Prix/m²',
               line=dict(color='#3b82f6', width=2)),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=monthly['mois_str'], y=monthly['nb_ventes'],
           name='Nb ventes', marker_color='#22c55e'),
    row=2, col=1
)

fig.update_layout(
    title="📈 Évolution mensuelle du marché immobilier",
    height=600, template='plotly_white', showlegend=False
)
fig.show()

In [8]:
# ============================================
# 🌡️ SAISONNALITÉ DES PRIX
# ============================================

if 'saison' in df.columns:
    saison_order = ['Hiver', 'Printemps', 'Été', 'Automne']
    df_saison = df[df['saison'].isin(saison_order)]

    fig = px.box(df_saison, x='saison', y='prix_m2',
                 color='saison',
                 category_orders={'saison': saison_order},
                 color_discrete_map={
                     'Hiver': '#60a5fa', 'Printemps': '#4ade80',
                     'Été': '#fbbf24', 'Automne': '#f97316'
                 },
                 title='🌡️ Saisonnalité des prix/m²')

    fig.update_layout(height=450, template='plotly_white', showlegend=False)
    fig.update_yaxes(title='Prix/m² (€)', range=[0, 15000])
    fig.show()

    print("\n📊 Prix médian par saison:")
    print(df_saison.groupby('saison')['prix_m2'].median().reindex(saison_order).to_string())


📊 Prix médian par saison:
saison
Hiver        4096.390
Printemps    4120.750
Été          4147.605
Automne      4148.590


In [9]:
# ============================================
# 🔗 MATRICE DE CORRÉLATIONS
# ============================================

corr_cols = ['valeur_fonciere', 'prix_m2', 'surface_reelle_bati', 'nb_pieces',
             'distance_centre_ville', 'commune_population', 'commune_densite']

# Garder seulement les colonnes qui existent
corr_cols = [c for c in corr_cols if c in df.columns]
corr_matrix = df[corr_cols].corr().round(2)

fig = px.imshow(corr_matrix,
                text_auto=True,
                color_continuous_scale='RdBu_r',
                title='🔗 Matrice de corrélations',
                aspect='auto')

fig.update_layout(height=500, template='plotly_white')
fig.show()

In [10]:
# ============================================
# 📐 SURFACE vs PRIX
# ============================================

df_sample = df.dropna(subset=['surface_reelle_bati', 'valeur_fonciere', 'type_local'])
df_sample = df_sample[df_sample['type_local'].isin(['Appartement', 'Maison'])]
df_sample = df_sample.sample(min(5000, len(df_sample)))

fig = px.scatter(df_sample,
                 x='surface_reelle_bati', y='valeur_fonciere',
                 color='type_local',
                 color_discrete_map={'Appartement': '#3b82f6', 'Maison': '#22c55e'},
                 opacity=0.5,
                 title='📐 Relation Surface vs Prix',
                 labels={
                     'surface_reelle_bati': 'Surface (m²)',
                     'valeur_fonciere': 'Prix (€)',
                     'type_local': 'Type'
                 })

fig.update_layout(height=500, template='plotly_white')
fig.show()

In [11]:
# ============================================
# 🏆 TOP COMMUNES
# ============================================

if 'nom_commune' in df.columns:
    top_communes = df.groupby('nom_commune').agg({
        'prix_m2': 'median',
        'valeur_fonciere': 'median',
        'id': 'count'
    }).rename(columns={'id': 'nb_ventes'})

    # Top 20 par nombre de ventes (minimum 10 ventes)
    top20 = top_communes[top_communes['nb_ventes'] >= 10].sort_values('prix_m2', ascending=False).head(20)

    fig = px.bar(top20.reset_index(),
                 x='prix_m2', y='nom_commune',
                 orientation='h',
                 color='prix_m2',
                 color_continuous_scale='Reds',
                 title='🏆 Top 20 communes par prix/m² (min 10 ventes)',
                 labels={'prix_m2': 'Prix/m² médian (€)', 'nom_commune': 'Commune'})

    fig.update_layout(height=600, template='plotly_white', yaxis={'categoryorder': 'total ascending'})
    fig.show()

In [12]:
# ============================================
# 💎 CATÉGORIES DE PRIX
# ============================================

if 'categorie_prix' in df.columns:
    cat_order = ['Bas', 'Moyen', 'Élevé', 'Premium', 'Luxe']
    cat_stats = df.groupby('categorie_prix').agg({
        'valeur_fonciere': ['count', 'median'],
        'prix_m2': 'median',
        'surface_reelle_bati': 'median'
    }).round(0)

    fig = px.pie(df[df['categorie_prix'].isin(cat_order)],
                 names='categorie_prix',
                 title='💎 Répartition par catégorie de prix',
                 color='categorie_prix',
                 color_discrete_map={
                     'Bas': '#22c55e', 'Moyen': '#3b82f6',
                     'Élevé': '#f59e0b', 'Premium': '#f97316', 'Luxe': '#ef4444'
                 },
                 category_orders={'categorie_prix': cat_order})

    fig.update_layout(height=450, template='plotly_white')
    fig.show()

    print("\n📊 Détail par catégorie:")
    print(cat_stats.to_string())


📊 Détail par catégorie:
               valeur_fonciere             prix_m2 surface_reelle_bati
                         count     median   median              median
categorie_prix                                                        
Bas                       4108   120000.0   2432.0                49.0
Luxe                      1596  1048000.0  10421.0                93.0
Moyen                     4014   210000.0   3478.0                61.0
Premium                   2450   518000.0   6250.0                81.0
Élevé                     4016   325000.0   4521.0                72.0


In [13]:
# ============================================
# 📏 TAILLE vs PRIX/m²
# ============================================

if 'taille_categorie' in df.columns:
    taille_order = ['Studio', 'Petit', 'Moyen', 'Grand', 'Très grand']

    fig = px.box(df[df['taille_categorie'].isin(taille_order)],
                 x='taille_categorie', y='prix_m2',
                 color='taille_categorie',
                 category_orders={'taille_categorie': taille_order},
                 title='📏 Prix/m² selon la taille du bien')

    fig.update_layout(height=450, template='plotly_white', showlegend=False)
    fig.update_yaxes(title='Prix/m² (€)', range=[0, 15000])
    fig.show()

In [14]:
# ============================================
# 📝 RÉSUMÉ ET INSIGHTS CLÉS
# ============================================

print("=" * 60)
print("📝 RÉSUMÉ DE L'ANALYSE EXPLORATOIRE")
print("=" * 60)

print(f"""
📊 DATASET:
  • {len(df):,} transactions immobilières
  • Période: {df['date_mutation'].min().strftime('%Y-%m-%d')} → {df['date_mutation'].max().strftime('%Y-%m-%d')}
  • Types: {df['type_local'].nunique()} types de biens

💰 PRIX:
  • Prix médian: {df['valeur_fonciere'].median():,.0f} €
  • Prix/m² médian: {df['prix_m2'].median():,.0f} €/m²
  • Surface médiane: {df['surface_reelle_bati'].median():,.0f} m²

🏠 PAR TYPE:
  • Appartements: {len(df[df['type_local']=='Appartement']):,} ({len(df[df['type_local']=='Appartement'])/len(df)*100:.0f}%)
  • Maisons: {len(df[df['type_local']=='Maison']):,} ({len(df[df['type_local']=='Maison'])/len(df)*100:.0f}%)

📍 GÉOGRAPHIE:
  • {df['nom_commune'].nunique() if 'nom_commune' in df.columns else 'N/A'} communes
  • {df['departement'].nunique() if 'departement' in df.columns else 'N/A'} départements

⭐ QUALITÉ:
  • Score moyen: {df['data_quality_score'].mean():.0f}/100
  • Taux géocodage: {df['latitude'].notna().mean()*100:.0f}%
""")

print("=" * 60)
print("✅ Analyse exploratoire terminée !")
print("=" * 60)

📝 RÉSUMÉ DE L'ANALYSE EXPLORATOIRE

📊 DATASET:
  • 16,184 transactions immobilières
  • Période: 2023-01-02 → 2023-12-06
  • Types: 2 types de biens

💰 PRIX:
  • Prix médian: 260,000 €
  • Prix/m² médian: 4,132 €/m²
  • Surface médiane: 65 m²

🏠 PAR TYPE:
  • Appartements: 10,822 (67%)
  • Maisons: 5,362 (33%)

📍 GÉOGRAPHIE:
  • 809 communes
  • 5 départements

⭐ QUALITÉ:
  • Score moyen: 95/100
  • Taux géocodage: 65%

✅ Analyse exploratoire terminée !
